# AntibodySearchEngine — local usage examples

This notebook shows how to call ABHunter’s Python API (`AntibodySearchEngine`) without writing SQL.
You pass normal search fields (V/D/J genes, CDR lengths, motifs) and the engine builds and runs the DuckDB queries.

**Returns of every `engine.search(...)` call**

| Object | Meaning |
|--------|--------|
| `results_df` | Sequence rows if `full_results=True`, otherwise same as `stats_df` |
| `stats_df` | Per-subject hit table (`subject`, `hits`, `total_sequences`, HPM, …) |
| `stats` | Summary dict (`total_hits`, search time, …) |

**Data:** these notebooks live in `publication/DemoNotebooks/`. Toggle `USE_FULL_DB` in setup:
- `False` (default): `examples/minimal/` under the package root
- `True`: `data/{Heavy,Light,Paired}` under the package root

**Raw SQL** is still possible via DuckDB directly — see `duckdb_sql_examples.ipynb` in this folder.

## 0. Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd

# publication/DemoNotebooks/ → package root
REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.search_engine import AntibodySearchEngine

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

# ---------------------------------------------------------------------------
# Data paths
# - DEMO (default): examples/minimal/{Heavy,Light,Paired}/Demo
# - FULL: data/{Heavy,Light,Paired}
# ---------------------------------------------------------------------------
USE_FULL_DB = False

DEMO_ROOT = REPO_ROOT / "examples" / "minimal"
DEMO_HEAVY = DEMO_ROOT / "Heavy" / "Demo"
DEMO_LIGHT = DEMO_ROOT / "Light" / "Demo"
DEMO_PAIRED = DEMO_ROOT / "Paired" / "Demo"

FULL_HEAVY = REPO_ROOT / "data" / "Heavy"
FULL_LIGHT = REPO_ROOT / "data" / "Light"
FULL_PAIRED = REPO_ROOT / "data" / "Paired"

if USE_FULL_DB:
    heavy_dir = FULL_HEAVY
    light_dir = FULL_LIGHT
    paired_dir = FULL_PAIRED
else:
    heavy_dir = DEMO_HEAVY
    light_dir = DEMO_LIGHT
    paired_dir = DEMO_PAIRED

print("USE_FULL_DB:", USE_FULL_DB)
print("heavy_dir:", heavy_dir.relative_to(REPO_ROOT), "exists=", heavy_dir.exists())
if light_dir is not None:
    print("light_dir:", light_dir.relative_to(REPO_ROOT), "exists=", light_dir.exists())
if paired_dir is not None:
    print("paired_dir:", paired_dir.relative_to(REPO_ROOT), "exists=", paired_dir.exists())

USE_FULL_DB: False
heavy_dir: examples/minimal/Heavy/Demo exists= True
light_dir: examples/minimal/Light/Demo exists= True
paired_dir: examples/minimal/Paired/Demo exists= True


In [2]:
# Initialize on unpaired heavy data (demo or full)
engine = AntibodySearchEngine(data_dir=str(heavy_dir), verbose=True)
print("search_type:", engine.schema.get("search_type"))
print("total_sequences:", f"{engine.total_sequences:,}")

DuckDB: memory_limit=40GB, threads=4
Registered 1 Parquet files
Total sequences: 50
search_type: unpaired
total_sequences: 50


## 1. Gene filters (V / D / J)

Accepted notations (same as the web form):
- family: `1`
- gene: `1-3`
- allele: `1-3*04`
- multiple: comma or pipe (`1-3,1-69` or `1-3|1-69`)
- light locus prefix: `L2` (lambda), `K2` (kappa), bare `2` (both)

The minimal Heavy demo is a single-gene subsample (`IGHV1-3*`).

In [3]:
# Single V gene (gene level)
results_df, stats_df, stats = engine.search(
    chain_mode="heavy",
    heavy_v="1-3",
)
print("total_hits:", stats["total_hits"])
display(stats_df.head())

total_hits: 50


,subject,total_sequences,hits,percentage,per_million
0,no,50,50,100.0,1000000.0


In [4]:
# Family-level V + specific J
_, stats_df, stats = engine.search(
    chain_mode="heavy",
    heavy_v="1",
    heavy_j="4",  # also accepts "J4"
)
print("IGHV1 × IGHJ4 hits:", stats["total_hits"])
display(stats_df.sort_values("hits", ascending=False).head(10))

IGHV1 × IGHJ4 hits: 18


,subject,total_sequences,hits,percentage,per_million
0,no,50,18,36.0,360000.0


In [5]:
# Multi-parameter VDJ
_, stats_df, stats = engine.search(
    chain_mode="heavy",
    heavy_v="1-3",
    heavy_j="4",
    heavy_cdr3_length="14",
)
print("IGHV1-3 + IGHJ4 + CDRH3=14 hits:", stats["total_hits"])
display(stats_df.head())

IGHV1-3 + IGHJ4 + CDRH3=14 hits: 6


,subject,total_sequences,hits,percentage,per_million
0,no,50,6,12.0,120000.0


## 2. CDR length filters

Length strings support:
- fixed: `"15"`
- range: `"12-18"`
- inequalities: `">14"`, `">=15"`, `"<20"`, `"<=18"`

In [6]:
_, stats_df, stats = engine.search(
    chain_mode="heavy",
    heavy_cdr3_length="15-20",
)
print("CDRH3 length 15–20 hits:", stats["total_hits"])
display(stats_df.head())

CDRH3 length 15–20 hits: 18


,subject,total_sequences,hits,percentage,per_million
0,no,50,18,36.0,360000.0


## 3. CDR motif filters

Motifs use the ABHunter simplified regex syntax (`*`, `.`, `[…]`, `{n}`, `{n,m}`, etc.).

- `similarity=False` (default): exact pattern match
- `similarity=True` + `mismatches>0`: allow similar amino acids at up to *k* defined positions

In [7]:
# Exact / wildcard motif
_, stats_df, stats = engine.search(
    chain_mode="heavy",
    heavy_v="1-3",
    heavy_cdr3_motif="AR*",
)
print("IGHV1-3 + CDRH3 motif AR* hits:", stats["total_hits"])
display(stats_df.head())

IGHV1-3 + CDRH3 motif AR* hits: 39


,subject,total_sequences,hits,percentage,per_million
0,no,50,39,78.0,780000.0


In [8]:
# Motif with 1 allowed similar substitution
_, stats_df, stats = engine.search(
    chain_mode="heavy",
    heavy_cdr3_motif="ARD",
    heavy_cdr3_similarity=True,
    heavy_cdr3_mismatches=1,
)
print("CDRH3 ARD (similarity, 1 mismatch) hits:", stats["total_hits"])
display(stats_df.head())

CDRH3 ARD (similarity, 1 mismatch) hits: 0


,subject,total_sequences,hits,percentage,per_million
0,no,50,0,0.0,0.0


## 4. Stats-only vs sequence retrieval

Default (`full_results=False`) is what precursor-frequency / web “statistics” mode uses.
Set `full_results=True` (optionally with `limit`) to materialize sequence rows.

In [9]:
# Statistics only (fast)
_, stats_df, stats = engine.search(
    chain_mode="heavy",
    heavy_v="1-3",
    full_results=False,
)
print("stats-only total_hits:", stats["total_hits"])
print("stats_df columns:", list(stats_df.columns))

# First 100 matching sequences (drop absolute path columns for cleaner display)
seqs, stats_df, stats = engine.search(
    chain_mode="heavy",
    heavy_v="1-3",
    full_results=True,
    limit=100,
)
print("returned sequence rows:", len(seqs))
display(seqs.drop(columns=[c for c in ("file_path", "source_file") if c in seqs.columns], errors="ignore").head())

stats-only total_hits: 50
stats_df columns: ['subject', 'total_sequences', 'hits', 'percentage', 'per_million']
returned sequence rows: 50


,v_call,d_call,j_call,sequence_alignment_aa,v_sequence_alignment_aa,v_germline_alignment_aa,d_sequence_alignment_aa,d_germline_alignment_aa,j_sequence_alignment_aa,j_germline_alignment_aa,cdr1_aa,cdr2_aa,cdr3_aa,cdr1_length,cdr2_length,cdr3_length,v_%SHM,d_%SHM,j_%SHM,filename,subject,chain,isotype,species,disease,vaccine
0,IGHV1-3*04,IGHD3-3*01,IGHJ6*02,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLA...,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLA...,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,YYDFWSGYY,YYDFWSGYY,YYYYGMDVWGQGTTVTVSS,YYYYGMDVWGQGTTVTVSS,GYTFTSYA,INTGNGNT,ARDTSSVKRLRPSSHTYYDFWSGYYSNYYYYGMDV,8,8,35,98.98,100.0,100.00,/Users/tomschlegel/Documents/ABH/antibody_sear...,no,Heavy,IGHM,human,None,None
1,IGHV1-3*04,IGHD3-3*01,IGHJ6*02,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,YYDFWSGYY,YYDFWSGYY,YYYYGMDVWGQGTTVTVSS,YYYYGMDVWGQGTTVTVSS,GYTFTSYA,INTGNGNT,ARDTSSVKRLRPSSHTYYDFWSGYYSNYYYYGMDV,8,8,35,100.00,100.0,100.00,/Users/tomschlegel/Documents/ABH/antibody_sear...,no,Heavy,IGHM,human,None,None
2,IGHV1-3*04,IGHD3-3*01,IGHJ6*02,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,YYDFWSGYY,YYDFWSGYY,YYYYGMDVWGQGTTDTVSS,YYYYGMDVWGQGTTVTVSS,GYTFTSYA,INTGNGNT,ERDTSSVKRLRPSSHTYYDFWSGYYSNYYYYGMDV,8,8,35,98.98,100.0,94.74,/Users/tomschlegel/Documents/ABH/antibody_sear...,no,Heavy,IGHM,human,None,None
3,IGHV1-3*04,IGHD3-3*01,IGHJ6*02,QVQLVQSGAEVKKPWASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,QVQLVQSGAEVKKPWASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,YYDFWSGYY,YYDFWSGYY,YYYYGMDVWGQGTTVTVSS,YYYYGMDVWGQGTTVTVSS,GYTFTSYA,INTGNGNT,ARDTSSVKRLRPSSHTYYDFWSGYYSNYYYYGMDV,8,8,35,98.98,100.0,100.00,/Users/tomschlegel/Documents/ABH/antibody_sear...,no,Heavy,IGHM,human,None,None
4,IGHV1-3*04,IGHD3-3*01,IGHJ6*02,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLE...,YYDFWSGYY,YYDFWSGYY,YYYYGMDVWGQGTTVTVSS,YYYYGMDVWGQGTTVTVSS,GYTFTSYA,INTGNGNT,ARDTSSVKRLRPSSHTYYDFWSGYYSNYYYYGMDV,8,8,35,97.96,100.0,100.00,/Users/tomschlegel/Documents/ABH/antibody_sear...,no,Heavy,IGHM,human,None,None


## 5. Light-chain and paired searches

Works with the minimal Demo folders, or set `USE_FULL_DB = True` for the full Light / Paired trees.

In [10]:
if light_dir is None or not Path(light_dir).exists():
    print("Skip light example: set USE_FULL_DB=True (or ensure DEMO light_dir exists).")
else:
    light_engine = AntibodySearchEngine(data_dir=str(light_dir), verbose=True)
    _, stats_df, stats = light_engine.search(
        chain_mode="light",
        light_v="L2",  # lambda family 2 only
    )
    print("IGLV2* hits:", stats["total_hits"])
    display(stats_df.head())
    light_engine.close()

DuckDB: memory_limit=40GB, threads=4
Registered 2 Parquet files
Total sequences: 35
IGLV2* hits: 14


,subject,total_sequences,hits,percentage,per_million
0,Subject-HIP3,22,11,50.00,500000.0
1,Subject-HIP2,13,3,23.08,230769.2


In [11]:
if paired_dir is None or not Path(paired_dir).exists():
    print("Skip paired example: set USE_FULL_DB=True (or ensure DEMO paired_dir exists).")
else:
    paired_engine = AntibodySearchEngine(data_dir=str(paired_dir), verbose=True)
    _, stats_df, stats = paired_engine.search(
        chain_mode="paired",
        heavy_v="4",
        heavy_cdr3_length="15-20",
    )
    print("paired IGHV4* × CDRH3 15–20 hits:", stats["total_hits"])
    display(stats_df.head())
    paired_engine.close()

DuckDB: memory_limit=40GB, threads=4


Registered 1 Parquet files
Total sequences: 40
paired IGHV4* × CDRH3 15–20 hits: 6


,subject,total_sequences,hits,percentage,per_million
0,417c,40,6,15.0,150000.0


## 6. Optional: same count via raw SQL

For comparison — AntibodySearchEngine wraps filters like this. Free-form aggregations
(e.g. allele co-occurrence matrices) are still easier as direct DuckDB SQL.

In [12]:
# Equivalent idea to engine.search(chain_mode="heavy", heavy_v="1-3") on this engine's view
n = engine.conn.execute(
    """
    SELECT COUNT(*)
    FROM antibodies
    WHERE v_call LIKE 'IGHV1-3%'
    """
).fetchone()[0]
print("raw SQL IGHV1-3* count:", n)

raw SQL IGHV1-3* count: 50


In [13]:
engine.close()
print("Done.")

Done.
